# Ingest races.csv file
### 1. Read the file using spark dataframe reader API
### 2. Add Metadata Columns 
-       Source File
-       Ingestion Timestamp
### 3. Write to bronze delta table

### Step 1 - Read the CSV file using the dataframe reader API

In [0]:
dbutils.widgets.text("p_batch_id", "")

In [0]:
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run  ../00-common/01_Environmnet_config

In [0]:
%run  ../00-common/02_bronze_helpers

In [0]:
source_file = f"{landing_folder_path}/{v_batch_id}/races.csv"
table_name = f"{catalog_name}.{bronze_schema}.races"
print(source_file)
print(table_name)

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DoubleType,
    DateType,
)

races_schema = StructType(
    [
        StructField("season", IntegerType()),
        StructField("round", IntegerType()),
        StructField("url", StringType()),
        StructField("raceName", StringType()),
        StructField("date", DateType()),
        StructField("circuitId", StringType()),
    ]
)

In [0]:
races_df = (
    spark.read.format("csv")
    .option("header", "true")
    .schema(races_schema)
    .option("mode", "FAILFAST")
    .load(source_file)
)

In [0]:
races_final_df = add_ingestion_medatat(races_df)

In [0]:
write_to_bronze(races_final_df, table_name, v_batch_id)

In [0]:
%sql
select * from formula1_incr.bronze.races
where batch_id = '2025-01'